# Hugging Face child table — extra ML check (ages 4–11)

This is **not** the main questionnaire result. The capstone primary set stays **UCI 292 AQ-10-Child** (`questionnaire.ipynb`).

We run the **same recipes** on Hugging Face V3 “child” after keeping ages **4–11** and dropping exact copies:

1. Can the 10 questions recover YES/NO? (on UCI this was almost perfect)
2. Can age/gender/ethnicity/jaundice do it without questions?
3. Do questions + background beat questions alone?

Expect mixed labels: many YES rows have a **low** question total. So we should **not** replace the 292-row model with this one.

In [ ]:
# Same libraries as the UCI questionnaire notebook.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Load Hugging Face child data

Prefer the **already filtered** file (ages 4–11, exact duplicates removed).
If you only have the raw HF child CSV (ages 1–11), we cut to 4–11 and drop copies here.

In [ ]:
candidates = [
    Path("/content/asd_screening_children_hf_v3_age4to11_dedup.csv"),
    Path("asd_screening_children_hf_v3_age4to11_dedup.csv"),
    Path("../data/raw/questionnaire/asd_screening_children_hf_v3_age4to11_dedup.csv"),
    Path("../data/raw/questionnaire/candidates/Child_ASD_Combined.csv"),
    Path("/content/Child_ASD_Combined.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Need asd_screening_children_hf_v3_age4to11_dedup.csv or Child_ASD_Combined.csv"
    )

df = pd.read_csv(data_path)
print("Loaded", data_path, "shape", df.shape)
print(df.columns.tolist())

# Raw HF names -> same names as the UCI notebook
rename = {
    **{f"A{i}": f"A{i}_Score" for i in range(1, 11)},
    "Age": "age",
    "ASD_label": "class",
}
df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

item_cols = [f"A{i}_Score" for i in range(1, 11)]
if "age" in df.columns:
    age = pd.to_numeric(df["age"], errors="coerce")
    df = df.loc[age.between(4, 11)].copy()

df = df.drop_duplicates()
df["class"] = df["class"].astype(str).str.strip().str.upper()
print("After 4–11 filter + dedup:", df.shape)
print(df["class"].value_counts().to_dict())
df.head()

## 1. Do YES/NO follow the 10-question total?

On UCI, YES ≈ high sum. If many YES children here have a **low** sum, this is not AQ-10-Child.

In [ ]:
df["aq_sum"] = df[item_cols].sum(axis=1)

print(df.groupby("class")["aq_sum"].describe())
print("\nCounts at each total:")
print(pd.crosstab(df["aq_sum"], df["class"], margins=True))

for cut in (6, 7):
    print(f"\nIf YES meant total >= {cut}:")
    print(pd.crosstab(df["aq_sum"] >= cut, df["class"], margins=True))

plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="aq_sum", hue="class", multiple="dodge", discrete=True)
plt.title("Question total by class (HF 4–11) — overlap means mixed labels")
plt.tight_layout()
plt.show()

## 2. Clean-up

Drop the reconstructed total (`aq_sum` / `score`) so the model cannot cheat with the sum column. The 10 items stay. Tidy text; missing ethnicity → `missing`.

In [ ]:
model_df = df.copy()
model_df = model_df.replace("?", np.nan)

for col in ["ethnicity", "gender", "jaundice"]:
    if col in model_df.columns:
        model_df[col] = (
            model_df[col].astype("string").str.strip().str.lower().fillna("missing")
        )

y = model_df["class"].map({"NO": 0, "YES": 1})
if y.isna().any():
    raise ValueError(y.value_counts(dropna=False))

# Leakage columns: never train on the sum
X_all = model_df.drop(columns=["class", "aq_sum", "score", "result"], errors="ignore")
demo_cols = [c for c in X_all.columns if c not in item_cols]

print("Items:", item_cols)
print("Background:", demo_cols)
print("Missing still:", X_all.isna().sum()[X_all.isna().sum() > 0].to_dict())
print("class", y.value_counts().to_dict())

## 3. Three feature sets + models

| Set | Meaning |
|---|---|
| **items** | 10 questions only |
| **demographics** | age, gender, ethnicity, jaundice |
| **both** | questions + background |

Models (no neural net — still a small table): logistic regression, short tree, random forest, histogram boosting.

Nested CV: outer 5-fold = the score we report; inner 3-fold = pick settings. Holdout 20% is only a peek.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)
print("Train", len(X_train), y_train.value_counts().to_dict())
print("Test ", len(X_test), y_test.value_counts().to_dict())


def make_preprocessor(feature_names):
    num = [c for c in feature_names if pd.api.types.is_numeric_dtype(X_all[c])]
    cat = [c for c in feature_names if c not in num]
    parts = []
    if num:
        parts.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                num,
            )
        )
    if cat:
        parts.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                cat,
            )
        )
    return ColumnTransformer(parts)


MODEL_GRIDS = {
    "log_reg": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {"model__C": [0.1, 1.0, 10.0]},
    ),
    "tree": (
        DecisionTreeClassifier(random_state=42, class_weight="balanced"),
        {"model__max_depth": [2, 3, 4]},
    ),
    "rf": (
        RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=200),
        {"model__max_depth": [3, 5]},
    ),
    "hgb": (
        HistGradientBoostingClassifier(random_state=42, max_depth=3),
        {"model__learning_rate": [0.05, 0.1]},
    ),
}

FEATURE_SETS = {
    "items": item_cols,
    "demographics": demo_cols,
    "both": list(X_all.columns),
}

In [ ]:
# Takes about 1 minute.
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rows = []
fitted = {}
for set_name, cols in FEATURE_SETS.items():
    Xtr = X_train[cols]
    for model_name, (est, grid) in MODEL_GRIDS.items():
        gs = GridSearchCV(
            Pipeline([("preprocess", make_preprocessor(cols)), ("model", est)]),
            grid,
            cv=inner,
            scoring="roc_auc",
            n_jobs=1,
            refit=True,
        )
        aucs = []
        for tr_idx, te_idx in outer.split(Xtr, y_train):
            gs.fit(Xtr.iloc[tr_idx], y_train.iloc[tr_idx])
            proba = gs.predict_proba(Xtr.iloc[te_idx])[:, 1]
            aucs.append(roc_auc_score(y_train.iloc[te_idx], proba))
        gs.fit(Xtr, y_train)
        fitted[(set_name, model_name)] = gs
        rows.append(
            {
                "feature_set": set_name,
                "model": model_name,
                "cv_mean": float(np.mean(aucs)),
                "cv_std": float(np.std(aucs)),
                "best_params": gs.best_params_,
            }
        )
        print(f"{set_name:13} {model_name:8}  {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

cv_table = pd.DataFrame(rows).sort_values(["feature_set", "cv_mean"], ascending=[True, False])
cv_table

## 4. Test-set results + flags in a world where YES is rare (1% and 2%)

Pick the best model per recipe using the nested-CV score, then look at the 20% holdout.
The 1%/2% numbers answer: if only 1 in 100 children should be YES, how often is a flag still a false alarm?

In [ ]:
def ppv_npv(sens, spec, p):
    ppv = (sens * p) / (sens * p + (1 - spec) * (1 - p) + 1e-12)
    npv = (spec * (1 - p)) / (spec * (1 - p) + (1 - sens) * p + 1e-12)
    return ppv, npv


summary_rows = []
for set_name in FEATURE_SETS:
    best_name = cv_table[cv_table["feature_set"] == set_name].iloc[0]["model"]
    cols = FEATURE_SETS[set_name]
    gs = fitted[(set_name, best_name)]
    y_pred = gs.predict(X_test[cols])
    y_prob = gs.predict_proba(X_test[cols])[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    rec = {
        "feature_set": set_name,
        "model": best_name,
        "test_correct_pct": (y_pred == y_test).mean(),
        "caught_YES": sens,
        "correct_NO": spec,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }
    for p, lab in ((0.01, "1pct"), (0.02, "2pct")):
        ppv, npv = ppv_npv(sens, spec, p)
        rec[f"flag_is_true_at_{lab}"] = ppv
    summary_rows.append(rec)
    print("\n===", set_name, "best:", best_name, "===")
    print(classification_report(y_test, y_pred, target_names=["NO", "YES"]))
    print("confusion [[true NO kept, false YES], [missed YES, true YES caught]]:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

screen_table = pd.DataFrame(summary_rows)
screen_table

## 5. Which questions the model uses (SHAP)

Fit logistic regression on the **10 items only**. Bars = average influence. This is for comparison with UCI, not a parent-facing product on this mixed table.

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap

item_lr = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
item_lr.fit(X_train[item_cols], y_train)

explainer = shap.LinearExplainer(item_lr, X_train[item_cols])
shap_values = explainer(X_test[item_cols])

mean_abs = pd.Series(np.abs(shap_values.values).mean(axis=0), index=item_cols).sort_values(
    ascending=False
)
print("Average |influence| per question:")
print(mean_abs)

shap.plots.bar(shap_values, show=True)
shap.plots.beeswarm(shap_values, show=True)

## How to read this vs the UCI 292 notebook

| | UCI 292 (main) | This HF 4–11 file |
|---|---|---|
| What YES means | almost “high AQ-10 total” | mixed — many YES with low totals |
| Questions-only models | near-perfect copy of the test | good (~85%) but **not** a clean copy |
| Background only | near a coin flip | still weak |
| Use in the capstone | **primary** questionnaire stream | **audit only** — do not replace 292 |

If questions-only is far from perfect here, the larger table is **not** “more AQ-10-Child.” Keep UCI for the proposal model.